# ⚡ MoneyPrinter Turbo - Kaggle GPU Video Üretim Stüdyosu

> **NVIDIA GPU Donanım Hızlandırmalı (NVENC), Google Drive & Kaggle Storage Entegre ve Kesintisiz Auto-Resume Destekli Video Üretim Sistemi**

> 💡 **Google Colab Kullanıcıları:** Google Colab ortamı için özel hazırlanmış sürüm: [`MoneyPrinterTurbo_Colab.ipynb`](MoneyPrinterTurbo_Colab.ipynb). *(Not: Gelecekteki güncellemelerde Colab ve Kaggle sürümleri senkronize tutulmalıdır.)*

> ⚠️ **Kaggle İçin Önemli Not:** Sağdaki **Notebook Options** panelinden **Accelerator -> GPU (T4 x2 veya P100)** ve **Internet -> On** olarak seçildiğinden emin olun.

Bu Kaggle Notebook ile MoneyPrinter Turbo Lite sürümünü çalıştırabilir; **NVIDIA GPU donanım hızlandırma ile saniyeler içinde 4K/1080p video üretebilir**, ayarlarınızı ve biten videolarınızı doğrudan Kaggle depolama alanında ve isteğe bağlı olarak **Google Drive** üzerinde saklayabilirsiniz.

### 🌟 Öne Çıkan Özellikler:
- 🚀 **Kaggle NVIDIA GPU (T4 / P100) NVENC Hızlandırma:** Render işlemleri CPU'ya kıyasla **10x - 30x daha hızlı** gerçekleşir.
- 📁 **Kaggle Root & Google Drive Entegrasyonu:** Kaggle kök dizinine (`/kaggle/working`) tam uyumlu çalışma; isteğe bağlı Google Drive senkronizasyonu.
- 🔄 **Kaldığı Yerden Devam Etme (Auto-Resume):** Oturum kapansa bile görevler ve biten videolar `tasks_db.json` üzerinden korunur.
- 🌐 **Cloudflare / Ngrok Tüneli:** Tek tıkla tarayıcıdan erişilebilen WebUI arayüzü.
- 📝 **Toplu Headless Üretim:** WebUI olmadan metin dosyalarını otomatik videolaştırma.

### 📦 1. Adım: Depoyu Klonlama, Paketleri Yükleme ve Kaggle GPU NVENC Kurulumu
*Kaggle root dizinini (`/kaggle/working`) ayarlar, FFmpeg GPU (NVENC), Cloudflared ve Python bağımlılıklarını kurar.*

In [ ]:
# @title 📦 1. Adım: Kurulum & Kaggle GPU NVENC Yapılandırması
import os, sys, shutil

# 1. GPU Kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'CPU Modu Aktif'

# 2. Kaggle Çalışma Dizinine Geç ve Depoyu Klonla
REPO_DIR = '/kaggle/working/MoneyPrinterTurbo-Lite'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/TheOsmanYILDIRIM/MoneyPrinterTurbo-Lite.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Temel paketleri, Cloudflared ve PyDrive2 kur
!apt-get update -qq && apt-get install -y -qq ffmpeg xz-utils
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!pip install -q -r requirements-lite.txt pyngrok pydrive2 gdown

# 4. Kaggle GPU (T4/P100) ile tam uyumlu FFmpeg 7.1 NVENC binary'sini kur
!if command -v nvidia-smi &> /dev/null; then \
    echo '⚡ Kaggle GPU için NVENC uyumlu FFmpeg 7.1 yapılandırılıyor...'; \
    wget -q -nc -O /tmp/ffmpeg-nvenc.tar.xz https://github.com/BtbN/FFmpeg-Builds/releases/download/autobuild-2026-08-16-13-00/ffmpeg-n7.1.5-16-g9a4bb2c579-linux64-gpl-7.1.tar.xz && \
    mkdir -p /tmp/ffmpeg_extracted && \
    tar -xf /tmp/ffmpeg-nvenc.tar.xz -C /tmp/ffmpeg_extracted && \
    cp -f /tmp/ffmpeg_extracted/*/bin/ffmpeg /usr/local/bin/ && \
    cp -f /tmp/ffmpeg_extracted/*/bin/ffprobe /usr/local/bin/ && \
    chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe && \
    rm -rf /tmp/ffmpeg_extracted /tmp/ffmpeg-nvenc.tar.xz && \
    echo '✅ NVENC FFmpeg başarıyla yüklendi!'; \
fi

# 5. Depolama Ortamını ve GPU Encoder'ı Başlat
import drive_sync, lite_engine
STORAGE_PATH = '/kaggle/working/MoneyPrinterTurbo'
drive_sync.init_drive_environment(drive_dir=STORAGE_PATH)
lite_engine._ENCODER_CONFIG = None
enc = lite_engine.get_video_encoder_config()
print(f"⚡ Aktif Video Encoder: {enc['name']} ({'NVIDIA GPU Hızlandırmalı 🚀' if enc.get('is_gpu') else 'CPU Modu'})")

### ☁️ 2. Adım: İsteğe Bağlı Google Drive Entegrasyonu & Senkronizasyon (Kaggle)
*Kaggle üzerinde üretilen videoları doğrudan Google Drive hesabınıza aktarmak veya Google Drive'daki ayar/görev dosyalarını Kaggle'a çekmek için bu adımı kullanabilirsiniz.*

In [ ]:
# @title ☁️ 2. Adım: Google Drive Senkronizasyon Aracı (İsteğe Bağlı)
import os

def sync_outputs_to_drive():
    """Kaggle outputs klasöründeki videoları Google Drive'a yüklemek için rehberlik sağlar."""
    outputs_dir = '/kaggle/working/MoneyPrinterTurbo/outputs'
    if not os.path.exists(outputs_dir):
        print('ℹ️ Henüz üretilmiş video bulunmuyor.')
        return
    
    vids = [f for f in os.listdir(outputs_dir) if f.endswith(('.mp4', '.mkv', '.mov'))]
    print(f'🎬 Kaggle Depolamasında {len(vids)} adet video mevcut: {outputs_dir}')
    for v in vids:
        fpath = os.path.join(outputs_dir, v)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'  • {v} ({size_mb:.2f} MB)')

sync_outputs_to_drive()

### ⚙️ 3. Adım: Ayarlar ve API Anahtarlarını Yapılandırma (İsteğe Bağlı)
*API anahtarlarınızı ve varsayılan video tercihlerini `settings.json` dosyasına kaydeder.*

In [ ]:
# @title ⚙️ 3. Adım: API Anahtarları ve Video Tercihleri
import settings_manager

PEXELS_API_KEY = ""
PIXABAY_API_KEY = ""
GEMINI_API_KEY = ""
OPENAI_API_KEY = ""
GROQ_API_KEY = ""
AZURE_SPEECH_KEY = ""
AZURE_SPEECH_REGION = "eastus"
ELEVENLABS_API_KEY = ""
NGROK_AUTHTOKEN = ""

DEFAULT_VOICE = "tr-TR-AhmetNeural"
DEFAULT_ASPECT = "9:16"
DEFAULT_RESOLUTION = "720p" # @param ["480p", "720p", "1080p", "2k", "4k"]
DEFAULT_BG_STYLE = "chalkboard"
SAVE_480P_DOWNGRADE = False # @param {type:"boolean"}

GPU_CQ = 23
GPU_PRESET = "p4"
CPU_CRF = 23
CPU_PRESET = "ultrafast"
AUDIO_BITRATE = "128k"

updates = {}
if PEXELS_API_KEY: updates["pexels_api_keys"] = PEXELS_API_KEY
if PIXABAY_API_KEY: updates["pixabay_api_keys"] = PIXABAY_API_KEY
if GEMINI_API_KEY: updates["gemini_api_key"] = GEMINI_API_KEY
if OPENAI_API_KEY: updates["openai_api_key"] = OPENAI_API_KEY
if GROQ_API_KEY: updates["groq_api_key"] = GROQ_API_KEY
if AZURE_SPEECH_KEY: updates["azure_speech_key"] = AZURE_SPEECH_KEY
if AZURE_SPEECH_REGION: updates["azure_speech_region"] = AZURE_SPEECH_REGION
if ELEVENLABS_API_KEY: updates["elevenlabs_api_key"] = ELEVENLABS_API_KEY
if NGROK_AUTHTOKEN: updates["ngrok_authtoken"] = NGROK_AUTHTOKEN

if DEFAULT_VOICE: updates["prod_voice"] = DEFAULT_VOICE
if DEFAULT_ASPECT: updates["prod_aspect"] = DEFAULT_ASPECT
if DEFAULT_RESOLUTION: updates["prod_resolution"] = DEFAULT_RESOLUTION
updates["prod_save_480p"] = bool(SAVE_480P_DOWNGRADE)
if DEFAULT_BG_STYLE: updates["prod_bg_style"] = DEFAULT_BG_STYLE
if GPU_CQ: updates["ffmpeg_cq_gpu"] = int(GPU_CQ)
if GPU_PRESET: updates["ffmpeg_preset_gpu"] = GPU_PRESET
if CPU_CRF: updates["ffmpeg_crf_cpu"] = int(CPU_CRF)
if CPU_PRESET: updates["ffmpeg_preset_cpu"] = CPU_PRESET
if AUDIO_BITRATE: updates["ffmpeg_audio_bitrate"] = AUDIO_BITRATE

if updates:
    settings_manager.save_settings(updates)
    print("✅ Ayarlar başarıyla kaydedildi!")

print("\n📋 Güncel Ayarlar:")
for k, v in settings_manager.get_masked_settings().items():
    if v:
        print(f"  • {k}: {v}")

### 🌐 4. Adım: WebUI Studio Sunucusunu Başlatma (Önerilen)
*Cloudflare veya Ngrok tüneli üzerinden WebUI arayüzünü açar. Tarayıcınızdan tekli ve toplu video üretimlerini kolayca yönetebilirsiniz.*

In [ ]:
# @title 🌐 4. Adım: WebUI Başlatıcı
TUNNEL_TYPE = "cloudflare"  # "cloudflare" veya "ngrok"
AUTO_RESUME = True

resume_flag = "--auto-resume" if AUTO_RESUME else ""
!python lite_server.py --host 0.0.0.0 --port 8080 --tunnel {TUNNEL_TYPE} {resume_flag} --storage-dir "/kaggle/working/MoneyPrinterTurbo"

### 🚀 5. Adım: Toplu Headless Video Üretimi (WebUI Olmadan)
*`/kaggle/working/MoneyPrinterTurbo/batch_inputs/` klasöründeki ders metinlerini sırayla render eder.*

In [ ]:
# @title 🚀 5. Adım: Toplu Headless Render & Auto-Resume
!python batch_processor.py

### 📊 6. Adım: Durum Raporu ve 1-Tık İndirme (Zip)
*Üretilen videoları listeler ve tüm biten videoları tek bir .zip arşivi haline getirerek Kaggle Output sekmesinden anında indirmenizi sağlar.*

In [ ]:
# @title 📊 6. Adım: Çıktıları Düzenle, Durum Raporu & Zip Olarak Paketle
import drive_sync, shutil, os
drive_sync.organize_drive()
drive_sync.print_drive_status()

storage_dir = drive_sync.get_default_storage_dir()
outputs_dir = os.path.join(storage_dir, 'outputs')
down_dir = os.path.join(storage_dir, 'downgraded_outputs')
zip_target = '/kaggle/working/MoneyPrinterTurbo_Outputs'
if os.path.exists(outputs_dir) or os.path.exists(down_dir):
    shutil.make_archive(zip_target, 'zip', storage_dir)
    print(f'📦 Tüm çıktılar (HD & 480p) zip olarak paketlendi: {zip_target}.zip')
    print('💡 Bu dosyayı sağdaki "Output" panelinden tek tıkla indirebilirsiniz!')
